In [0]:
%pip install VarClusHi==0.1.0
dbutils.library.restartPython()

In [0]:
%run ../../config/utils

In [0]:
import pyspark.sql.functions as f
import pandas as pd
import mlflow
from mlflow.client import MlflowClient
from mlflow.models.signature import infer_signature
from datetime import datetime

In [0]:
run_date_str = dbutils.widgets.get('run_as_date')
TENURE_GROUP = dbutils.widgets.get('tenure_group') if dbutils.widgets.get('tenure_group') in ['tenured', 'new'] else 'tenured'
current_year = int(run_date_str[:4])

In [0]:
latest_run_date = spark.table(gm_etl_output).filter(f.col('RUN_DATE') <= run_date_str).agg(f.max('RUN_DATE').alias('max_dt')).first()['max_dt']
data = spark.table(gm_etl_output).filter((f.col('RUN_DATE') == latest_run_date) & (f.col('DATASET_CD') == 'train')).drop('RUN_DATE','DATASET_CD') # spark.read.parquet('s3://memberanalytics-data-out-prod/USERS/skarunanithi/GM_Propensity_Model/2025/GM_ETL_2025_09_22_MAILER_model_v2.parquet/')#
# data.groupby('GM_purchase').agg(f.mean('BJS_DISTANCE'),f.mean('COSTCO_DISTANCE'),f.mean('SAMS_DISTANCE'),f.mean('WALMART_DISTANCE'),f.mean('member_age'),f.mean('household_income')).toPandas()

In [0]:
# data.count()
# data.groupby('GM_purchase').agg(f.countDistinct('MBRSHP_SID')).show()
# Note that differences in counts are due to the fact thatetl setp is using a random samplig without any seed

In [0]:
data = data.filter(data.TENURE_GROUP==TENURE_GROUP)
# data.groupby('GM_PURCHASE').agg(f.countDistinct('MBRSHP_SID')).show()

In [0]:
#remove outliers
column = 'FW_SPEND_IN_STORE'
quantile = data.approxQuantile(column, [0.05, 0.95], 0.01)
#print(quantile)
data = data.filter((f.col(column) >= quantile[0]) & (f.col(column) <= quantile[1]))
# data.groupby('GM_PURCHASE').agg(f.countDistinct('MBRSHP_SID')).show()

In [0]:
data_pd = data.toPandas()
data_pd.columns = data_pd.columns.str.upper()

In [0]:
# Drop Date Columns
#dropping date columns
date_columns = []
for i in data_pd.select_dtypes(include='object').columns:
    try:
        pd.to_datetime(data_pd[i].mode())
        date_columns.append(i)
    except:
        pass

data_pd = data_pd.drop(date_columns, axis=1)
print('Dropped date columns')
data_pd.shape

In [0]:
#Remove columns with more than 20% of missing values
null_count = data_pd.isnull().sum()
null_cols = null_count[null_count / data_pd.shape[0] > 0.2].index
data_pd.drop(
    null_cols,
    axis=1,
    inplace=True
)
data_pd.shape

In [0]:
target_feature = ['GM_PURCHASE']
categorical_features = list(set(data_pd.select_dtypes(include=['object']).columns) - set(list(target_feature))- set(['LATEST_MBRSHP_NBR','MBR_PRMRY_SID','MBR_SID','MBRSHP_SID','LATEST_HOME_ZIP_CD']))
continious_features = list(set(data_pd.select_dtypes(exclude=['object']).columns) - set(list(target_feature)) - set(list(target_feature))- set(['LATEST_MBRSHP_NBR','MBR_PRMRY_SID','MBRSHP_SID','LATEST_HOME_ZIP_CD','MBR_SID']))
print(len(categorical_features)+len(continious_features)+len(target_feature))

In [0]:
#Standardizing the data type
data_pd = (
    pd.concat(
        objs=[
            data_pd['MBRSHP_SID'],
            data_pd[continious_features].astype(float), 
            data_pd[categorical_features].astype(str),
            data_pd[target_feature]
        ],
        axis=1,
    )
)

In [0]:
#Filling the missing values
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder,LabelEncoder

data_pd[categorical_features] =  data_pd[categorical_features].fillna('missing')

data_pd[categorical_features] =  data_pd[categorical_features].apply(LabelEncoder().fit_transform)

data_pd[continious_features] =  data_pd[continious_features].fillna(data_pd[continious_features].median())

In [0]:
# Correlation between continious variables and the target
from scipy.stats import chi2_contingency
import scipy.stats as stats
import numpy as np

continious_corr = []
for i in continious_features:
    if i != target_feature[0] and i not in ['LATEST_MBRSHP_NBR','MBR_PRMRY_SID','MBR_SID','MBRSHP_SID']:
        #print(df[i])
        #print(df[target_feature[0]])
        A = stats.pointbiserialr(np.array(list(data_pd[i])), np.array(list(data_pd[target_feature[0]])))
        try:
            continious_corr.append(A[0])
        except:
            continious_corr.append(0)

continious_correlation = pd.DataFrame()
continious_correlation['Variables'] = continious_features
continious_correlation['Correlation'] = continious_corr
continious_correlation['Mod_Correlation'] = continious_correlation['Correlation'].abs()
print("Performed Correlation")
continious_correlation.sort_values(by=['Mod_Correlation'], ascending=False).head(25)

In [0]:
continious_features = sorted(list(set(continious_features) & set(list(continious_correlation[continious_correlation['Mod_Correlation']>=0.10]['Variables']))))

In [0]:
# Correlation between categorical variables and the target
def cramers_v(x, y):
    confusion_matrix = pd.crosstab(x,y)
    chi2 = chi2_contingency(confusion_matrix)[0]
    n = confusion_matrix.sum().sum()
    phi2 = chi2/n
    r,k = confusion_matrix.shape
    def maximum(a, b):     
        if a >= b:
            return a
        else:
            return b
        
    def minimum(a, b):     
        if a <= b:
            return a
        else:
            return b
    phi2corr = maximum(0, phi2-((k-1)*(r-1))/(n-1))
    rcorr = r-((r-1)**2)/(n-1)
    kcorr = k-((k-1)**2)/(n-1)
    return np.sqrt(phi2corr/minimum((kcorr-1),(rcorr-1)))

categorical_corr = []
for i in categorical_features:
    if i != target_feature[0] and i not in ['LATEST_MBRSHP_NBR','MBR_PRMRY_SID','MBR_SID','MBRSHP_SID']:
        cat_cor = cramers_v(np.array(list(data_pd[i])), np.array(list(data_pd[target_feature[0]])))
        try:
            categorical_corr.append(cat_cor)
        except:
            categorical_corr.append(0)
            
categorical_correlation = pd.DataFrame()
categorical_correlation['Variables'] = categorical_features
categorical_correlation['Correlation'] = categorical_corr
categorical_correlation['Mod_Correlation'] = categorical_correlation['Correlation'].abs()
print("Performed Correlation")
categorical_correlation.sort_values(by=['Mod_Correlation'], ascending=False).head(25)

In [0]:
categorical_features = sorted(list(set(categorical_features) & set(list(categorical_correlation[categorical_correlation['Mod_Correlation']>=0.10]['Variables']))))

In [0]:
from varclushi import VarClusHi
demo1_vc = VarClusHi(data_pd[continious_features+categorical_features],maxeigval2=1,maxclus=None)
demo1_vc.varclus()

In [0]:
demo1_vc.info

In [0]:
demo1_vc.rsquare.head()

In [0]:
data_pd_v2 = data_pd[['GM_PURCHASE','MBRSHP_SID']+list(demo1_vc.rsquare.sort_values(['Cluster','RS_Ratio'],ascending=True).groupby('Cluster').first().reset_index()['Variable'])]
data_pd_v2.head()

In [0]:
continious_features = sorted(list(set(continious_features) & set(list(data_pd_v2.columns))))
categorical_features = sorted(list(set(categorical_features) & set(list(data_pd_v2.columns))))

categorical_features = list(map(str.upper,categorical_features))
continious_features = list(map(str.upper,continious_features))

In [0]:
correlation = pd.concat([categorical_correlation, continious_correlation])
correlation_v2 = correlation[correlation['Variables'].isin(continious_features+categorical_features)]
correlation_v2.sort_values(['Mod_Correlation'],ascending=False)

In [0]:
df_dataset = spark.createDataFrame(data_pd_v2).withColumn('RUN_DATE', f.lit(run_date_str)).withColumn('TENURE_GROUP_FILTER', f.lit(TENURE_GROUP))
df_dataset.write.mode('overwrite').option('replaceWhere', f"RUN_DATE = '{run_date_str}' AND TENURE_GROUP_FILTER = '{TENURE_GROUP}'").option('mergeSchema', True).saveAsTable(gm_dataset)

In [0]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder


numeric_transformer = Pipeline(
    steps=[
        ("imputer_num", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer_cat", SimpleImputer(strategy="constant")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, continious_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [0]:
experiment_name = experiment_name_gm_model

mlflow.sklearn.autolog(disable=True)

mlflow.set_tracking_uri("databricks")
mlflow.set_registry_uri('databricks-uc')

if mlflow.get_experiment_by_name(experiment_name) is None:
    mlflow.create_experiment(name=experiment_name) 
mlflow.set_experiment(experiment_name)

In [0]:
mark_datetime = datetime.strftime(datetime.now(), '_%Y-%m-%d_%H-%M-%S')
with mlflow.start_run(run_name=f'training_gm_preprocessor_{TENURE_GROUP}_{mark_datetime}') as run:
    data_pd_v3 = data_pd_v2.copy()
    data_pd_v3.columns = data_pd_v3.columns.str.upper()
    #preprocess.fit(df_sample)
    processed = preprocess.fit_transform(data_pd_v3)

    example_in = data_pd_v3.sample(5).copy()
    int_cols = example_in.select_dtypes(include=["int", "int32", "int64"]).columns
    example_in[int_cols] = example_in[int_cols].astype("float64")

    example_out = preprocess.transform(example_in)
    # if OHE is sparse, make a small dense sample only for signature
    if hasattr(example_out, "toarray"):
        example_out_sig = example_out.toarray()
    else:
        example_out_sig = example_out

    signature = infer_signature(example_in, example_out_sig)

    mlflow.log_params(preprocess.get_params())
    mlflow.log_param('ColumnTransformer', preprocess.named_transformers_)
    mlflow.set_tags({
        'estimator_class': 'sklearn.compose.ColumnTransformer',
        'estimator_name': 'ColumnTransformer'        
    })

    model_info = mlflow.sklearn.log_model(
        sk_model=preprocess,
        artifact_path="preprocessor",
        signature=signature,
        input_example=example_in,
        registered_model_name=gm_preprocessing_catalog,
    )
    


In [0]:
client = MlflowClient()
client.set_registered_model_alias(name=gm_preprocessing_catalog, alias=TENURE_GROUP, version=model_info.registered_model_version)